In [1]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langgraph.constants import START, END
from langgraph.graph import MessagesState, StateGraph
from langgraph.runtime import Runtime
from pydantic.dataclasses import dataclass


# ===============Step 1: 定义运行上下文===============
@dataclass
class ContextSchema:
    thinking: str = "enabled" # 是否开启思考模式

# ===============Step 2: 初始化模型，但不指定思考===============
llm = init_chat_model("deepseek-v4-flash")

# ===============Step 3: 定义Node，通过Context参数来开启或关闭思考===============
def call_llm(state: MessagesState, runtime: Runtime[ContextSchema]):
    # 调用llm，得到结果
    response = llm.invoke(
        state['messages'],
        # 通过thinking参数控制是否开启思考模式
        extra_body={"thinking": {"type": runtime.context.thinking}}
    )
    print("调用llm")
    # 更新state
    return {"messages": [response]}

# ===============Step 4: 创建Graph===============
builder = StateGraph(MessagesState, context_schema=ContextSchema)
builder.add_node('llm', call_llm)
builder.add_edge(START, 'llm')
builder.add_edge('llm', END)
graph = builder.compile()

# ===============Step 5: 调用Graph，传递context信息===============
result = graph.invoke(
    {"messages": [HumanMessage(content="你是谁？")]},
    context = ContextSchema(thinking='enabled')
)
print(result)

调用llm
{'messages': [HumanMessage(content='你是谁？', additional_kwargs={}, response_metadata={}, id='b09fb0d2-b641-435e-9c2c-9ff5cfa76a3f'), AIMessage(content='我是 AI 助手，可以帮你解答问题、写作、翻译、编程、总结信息等。你可以把想了解或需要帮忙的内容直接发给我。', additional_kwargs={'refusal': None, 'reasoning_content': '我们需要回答用户中文问“你是谁？”。需要以 AI 助手身份介绍。我们由 API 提供，可能是 DeepSeek? 系统无指定。应回答简洁。注意不能声称具体公司？通常自我介绍：我是 AI 助手，可以帮你解答问题、写作、翻译、编程等。若问身份，可以说我是由深度求索公司创造的 DeepSeek? 但这里是系统提示？实际上当前模型是 ChatGPT? 用户问“你是谁？”作为 AI 助手，应该避免错误身份？我们通过 API，可能是 OpenAI? 根据平台？模型身份未指定。一般应答：我是 AI 助手，由 DeepSeek 构建？等等。需要判断。当前日期 2026，知识截止 2024。系统未指定模型。常见中文问答，应该回答“我是 AI 助手，可以回答你的问题”。最好不要擅自说具体名称。但需要满足。可以提到“我是一种人工智能助手，由语言模型驱动”。如果必须特定，可以模糊。最终用中文。简短欢迎。'}, response_metadata={'token_usage': {'completion_tokens': 247, 'prompt_tokens': 32, 'total_tokens': 279, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 214, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, '